In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [2]:
from llama3_2.config import LlamaConfig
from llama3_2.model_trfmrs import LlamaModel
import safetensors.torch as st

vocab_size = 32768
student_config = LlamaConfig(vocab_size=vocab_size)
student_model = LlamaModel(student_config)
student_model = student_model.to(device)


In [3]:
student_model_states = st.load_file("student_model.safetensors")
student_model.load_state_dict(student_model_states)
student_model

LlamaModel(
  (embed_tokens): Embedding(32768, 2048)
  (layers): ModuleList(
    (0-15): 16 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (k_proj): Linear(in_features=2048, out_features=512, bias=False)
        (v_proj): Linear(in_features=2048, out_features=512, bias=False)
        (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)

In [4]:
# freeze student model except embed_tokens
for name, param in student_model.named_parameters():
    if name != "embed_tokens.weight":
        param.requires_grad = False
    else:
        param.requires_grad = True
        print(f"Keeping {name} trainable")


Keeping embed_tokens.weight trainable


In [5]:
import pandas as pd
df = pd.read_pickle("token_to_embeddings.pkl")
# sort by token_id
df = df.sort_values(by="token_id")
df

,token,token_id,embedding
10452,<uppercase>,0,"[0.14480376243591309, -0.15351225435733795, 2...."
10453,<space>,1,"[0.14459004998207092, -0.15334707498550415, 2...."
10454,<newline>,2,"[0.14467868208885193, -0.1533983200788498, 2.1..."
10455,<tab>,3,"[0.14478552341461182, -0.1534707397222519, 2.1..."
10456,<unknown>,4,"[0.1447853147983551, -0.15350626409053802, 2.1..."
...,...,...,...
10194,bpe_32763,32763,"[-0.5330702066421509, 3.6590523719787598, 2.37..."
10195,bpe_32764,32764,"[-0.0499199740588665, 3.5398025512695312, 2.42..."
10196,bpe_32765,32765,"[-0.5490577816963196, 3.595024824142456, 2.536..."
10197,bpe_32766,32766,"[-0.7619533538818359, 3.5637688636779785, 2.82..."


In [15]:
v = student_model(torch.tensor([[10000]]).to(device))
v.shape

torch.Size([1, 1, 2048])

In [23]:
from torch.utils.data import Dataset, DataLoader
import numpy as np

class CustomDataset(Dataset):
    def __init__(self, data_df: pd.DataFrame, batch_size: int):
        self.data_df = data_df
        self.batch_size = batch_size
        self.data_size = len(data_df)
        
        # Convert pandas Series to numpy arrays first, then to tensors
        self.input_ids = torch.tensor(data_df['token_id'].values, dtype=torch.long)
        
        # Convert embedding lists to numpy array, then to tensor
        embeddings = np.array(data_df['embedding'].tolist())
        self.targets = torch.tensor(embeddings, dtype=torch.float)

    def __len__(self):
        return self.data_size
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.targets[idx]

def create_dataloader(data_df: pd.DataFrame, batch_size: int):
    dataset = CustomDataset(data_df, batch_size)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataloader

In [26]:
train_dataloader = create_dataloader(df, batch_size=256)
len(train_dataloader)

123

In [69]:
import torch.nn as nn

loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(student_model.parameters(), lr=1e-4)

In [70]:
import tqdm

epochs = 100

student_model.train()

for epoch in range(epochs):    
    total_loss = 0
    progress_bar = tqdm.tqdm(train_dataloader, desc=f"Epoch {epoch+1}")
    
    for batch in progress_bar:
        input_ids, targets = batch
        input_ids = input_ids.to(device)
        targets = targets.to(device)
        v = student_model(input_ids.unsqueeze(0))
        optimizer.zero_grad()
        loss = loss_fn(v.squeeze(0), targets)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        
        # Update progress bar with current batch loss and average loss
        avg_loss = total_loss / (progress_bar.n + 1)  # Calculate running average
        progress_bar.set_postfix({'batch loss': f'{loss.item():.4f}', 'avg loss': f'{avg_loss:.4f}'})
    
    if (epoch + 1) % 50 == 0:
        # save embedding
        torch.save(student_model.state_dict(), f"student_model_{epoch}_{avg_loss:.4f}.pth")
    
    # Calculate final average loss for the epoch
    avg_loss = total_loss / len(train_dataloader)

Epoch 100: 100%|██████████| 123/123 [01:09<00:00,  1.78it/s, batch loss=1.0044, avg loss=0.8380]


In [64]:
student_model.state_dict()

OrderedDict([('embed_tokens.weight',
              tensor([[ 0.7739, -1.1908,  1.1909,  ...,  1.3665,  0.1002,  0.6520],
                      [ 0.0325, -0.4806,  0.3174,  ...,  1.1908,  0.1851,  0.5876],
                      [ 0.6140, -0.9606, -0.0984,  ..., -0.6666, -0.1290, -0.0285],
                      ...,
                      [-0.6739,  0.2665, -0.5365,  ...,  0.8167,  0.4310,  0.5561],
                      [ 1.2002,  1.8110, -0.7420,  ..., -1.2365,  0.9158,  0.3853],
                      [-0.6810,  0.1419,  0.2711,  ...,  0.3492, -0.3333,  0.6768]],
                     device='mps:0')),
             ('layers.0.self_attn.q_proj.weight',
              tensor([[-0.0183,  0.0071,  0.0219,  ..., -0.0070, -0.0089,  0.0149],
                      [ 0.0112,  0.0593,  0.0630,  ..., -0.0334, -0.0148,  0.0058],
                      [ 0.0182,  0.0141,  0.0361,  ..., -0.0432, -0.0388, -0.0233],
                      ...,
                      [ 0.0305,  0.0289,  0.0801,  ..., -0.0767